#1. Two paper findings + my methodology questions
Finding 1

Paper finding: Growing content is generally longer and younger than declining content.

My methodology question:
How were the "growing" and "declining" labels defined? Were they based only on observed trend data, and were any features derived from the same trend calculation excluded from later ML models to avoid leakage?

Finding 2

Paper finding: Portfolio CTR decreases as average search position becomes worse.

My methodology question:
Were these results calculated using weighted CTR across all impressions, and does the comparison control for differences in page type or search intent before drawing conclusions?

In [4]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("content_refresh_anonymized.csv")

df["target"] = (df["trend_direction"] == "down").astype(int)

features = [
    "search_volume",
    "word_count",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update"
]

X = df[features]
y = df["target"]

X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=features)

groups = df["client_id"]

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

Accuracy: 0.530585753691384
Precision: 0.5337730870712402
Recall: 0.6424261670371546
F1: 0.5830811356103185


#3. Leakage Audit

Leakage Audit
The target variable is derived from trend_direction.
I did not use trend_direction or trend_pct as model features because they directly determine the target.
I also excluded identifiers such as content_id and client_id from the feature set. The client ID was used only for grouped validation.
Features were limited to information available before prediction, reducing the risk of label leakage.

#4. Claim Rewrite

Original claim

The Random Forest model predicts declining pages accurately.

Revised claim

The Random Forest model achieved the best performance among the tested models on this dataset. The results are observational and should be viewed as decision-support rather than proof that the selected features cause content decline.

✓ I reviewed two findings from the research paper and asked methodology questions.

✓ I re-ran my model using a grouped validation split based on client_id.

✓ I compared the grouped split against my previous random split.

✓ I checked for label leakage and removed trend_direction and trend_pct from the feature set.

✓ My conclusions use careful language such as observed, measured, and decision-support.